In [3]:
from dotenv import load_dotenv
from langchain_teddynote import logging as langsmith_logging

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import load_prompt
from langchain_core.runnables import RunnablePassthrough
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
# utils 모듈에서 모든 필요한 기능들을 import
from example.utils import (
    load_pdf_with_toc_filter,
    DebugUpstageAsymmetricEmbeddings,
    get_or_create_vector_store
)
from langchain_community.vectorstores import FAISS

# API KEY 정보로드
load_dotenv()
# Langsmith 로깅 설정
langsmith_logging.langsmith("RAG-EXAMPLE-02")

# 문서 파싱 및 목차 필터링
docs = load_pdf_with_toc_filter("./data/SPRI_AI_Brief_2023년12월호_F.pdf")

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"✅분할된 문서의 수: {len(split_docs)}")

# 임베딩 & 벡터스토어 저장
embeddings = DebugUpstageAsymmetricEmbeddings()
db = FAISS.from_documents(split_docs, embeddings)

# ChatOpenAI 사용
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# 프롬프트 로드
prompt = load_prompt("prompts/rag-prompts.yaml")


# 기존 함수도 유지 (호환성을 위해)
def format_docs(docs) -> str:
    """검색된 문서를 프롬프트용 문자열로 직렬화한다."""
    return "\n\n".join(
        f"<document><content>{doc.page_content}</content><page>{doc.metadata.get('page', 'Unknown')}</page><source>{doc.metadata.get('source', 'Unknown')}</source></document>"
        for doc in docs
    )

# 검색을 위해 임베딩
# query_embedder = DebugUpstageAsymmetricEmbeddings()
# db = get_or_create_vector_store(
#     embedding=query_embedder,
# )

# 리트리버 생성
# retriever = db.as_retriever(search_kwargs={"k": 10})
# # 리랭커 생성
# compressor = CohereRerank(model="rerank-multilingual-v3.0")
# compression_retriever = ContextualCompressionRetriever(
#     base_compressor=compressor,
#     base_retriever=retriever,
# )
# retriever = db.as_retriever()
# retriever = db.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={"score_threshold": 0.2, "k": 4},  # 이제 거리 기준으로 수정
# )
#
# # 체인 생성
# chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

LangSmith 추적을 시작합니다.
[프로젝트명]
RAG-EXAMPLE-02
✅파싱된 문서의 수: 23
✅목차 필터링 후 문서의 수: 23
✅내용 감소량: 1895 문자
✅분할된 문서의 수: 41
[DEBUG] embed_documents() 호출 → 모델: solar-embedding-1-large-passage


In [19]:
retriever = db.as_retriever(
    # 검색 유형을 "similarity_score_threshold 으로 설정
    search_type="similarity_score_threshold",
    # 임계값 설정
    search_kwargs={"score_threshold": 0.3},
)

# 관련 문서를 검색
for doc in retriever.invoke("삼성전자가 만든 생성형 AI 모델은 무엇인가요?"):
    print(doc.page_content)
    print("=========================================================")

[DEBUG] embed_query() 호출 → 모델: solar-embedding-1-large-query
SPRi AI Brief |
2023-12월호
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
KEY Contents
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
£언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델
‘삼성 가우스’를 최초 공개
∙ 정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에
최적화된 크기의 모델 선택이 가능
∙ 삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며,
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙ 삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는
이미지 모델의 3개 모델로 구성
∙ 언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의
처리를 지원
∙ 코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며
사내 소프트웨어 개발에 최적화
∙ 이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 